# **Library**

In [4]:
# !pip install -U transformers accelerate
# !pip install optimum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 65.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# Remove Warning
import warnings
warnings.filterwarnings("ignore")
# ===========================================================
import os
import re
# Install Torch with GPU [pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121]
import torch
import torch.nn as nn
# ===========================================================
# Dataset
from datasets import load_dataset
# ===========================================================
# Transformers
import transformers
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model,PeftModel,PeftConfig , prepare_model_for_kbit_training


# **LLM Model**

In [3]:
# Model
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    device_map="auto",
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
print(model.device)

cuda:0


In [5]:
# Tokenizer is Specific Model
tokinzer = AutoTokenizer.from_pretrained("microsoft/phi-2" , truncation=True , padding=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [6]:
prompt = "Find the describing words for the following sentence:\n\"After they had broken up, he wasn't the same\"."

batch = tokinzer(prompt, return_tensors="pt").to(model.device)

output_tokens = model.generate(
    **batch,
    max_new_tokens=50,
    repetition_penalty=1.1,
)

output = tokinzer.decode(output_tokens[0], skip_special_tokens=True)
print(output)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Find the describing words for the following sentence:
"After they had broken up, he wasn't the same".
Answer: After, they, had, broken, up, he, wasn't, the, same.



# **Trainable Parameters**

In [7]:
def prit_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _ , param in model.named_parameters():
        all_param += param.numel()
        print(param.shape , param.numel() , param.dtype )
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * (trainable_params/all_param)}")

In [8]:
print(prit_trainable_parameters(model))

torch.Size([51200, 2560]) 131072000 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([10240, 2560]) 26214400 torch.float16
torch.Size([10240]) 10240 torch.float16
torch.Size([2560, 10240]) 26214400 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([10240,

In [9]:
model

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_layernorm): LayerNorm((2560,), eps=1

# **Lora Config**

In [10]:
config = LoraConfig(
    r = 4 , #rank
    lora_alpha = 32 , #alpha [Scaling factor]
    target_modules = ["q_proj" , "k_proj" , "v_proj" , "out_proj"] , #target modules
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)
peft_model = get_peft_model(model , config)
prit_trainable_parameters(peft_model)


torch.Size([51200, 2560]) 131072000 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([4, 2560]) 10240 torch.float32
torch.Size([2560, 4]) 10240 torch.float32
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([4, 2560]) 10240 torch.float32
torch.Size([2560, 4]) 10240 torch.float32
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([4, 2560]) 10240 torch.float32
torch.Size([2560, 4]) 10240 torch.float32
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([10240, 2560]) 26214400 torch.float16
torch.Size([10240]) 10240 torch.float16
torch.Size([2560, 10240]) 26214400 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([2560, 2560]) 6553600 torch.float16
torch.Size([2560]) 2560 torch.float16
torch.Size([4, 2560])

# **Dataset**

In [11]:
data = load_dataset('Abirate/english_quotes')
data

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})

In [12]:
data.keys()

dict_keys(['train'])

In [13]:
data['train'][0]

{'quote': '“Be yourself; everyone else is already taken.”',
 'author': 'Oscar Wilde',
 'tags': ['be-yourself',
  'gilbert-perreira',
  'honesty',
  'inspirational',
  'misattributed-oscar-wilde',
  'quote-investigator']}

In [14]:
def merge_columns(example):
    example['predection'] = f"""
    Find the describing words for the following sentence:
    "{example['quote']}"
    Tags: {str(example['tags'])}
    """
    return example

data['train'] = data['train'].map(merge_columns)
data['train']['predection'][0]


Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

'\n    Find the describing words for the following sentence:\n    "“Be yourself; everyone else is already taken.”"\n    Tags: [\'be-yourself\', \'gilbert-perreira\', \'honesty\', \'inspirational\', \'misattributed-oscar-wilde\', \'quote-investigator\']\n    '

In [15]:
data = data.map(lambda sample : tokinzer(sample['predection']) , batched=True)
data

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags', 'predection', 'input_ids', 'attention_mask'],
        num_rows: 2508
    })
})

# **Fine Tunning The model**

In [16]:
tokinzer.pad_token = "[PAD]"

In [17]:
args = transformers.TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_steps=100,
    num_train_epochs=6,
    fp16=True,
    logging_steps=10,
    output_dir="outputs",
    save_steps=200,
    save_total_limit=2,
)
# Data Collator
data_collator = transformers.DataCollatorForLanguageModeling(
    tokinzer,
    mlm=False
)

In [18]:
trainer = transformers.Trainer(
    model = peft_model ,
    train_dataset = data['train'],
    args = args,
    data_collator = data_collator
)
peft_model.config.use_cache = False
trainer.train()

Step,Training Loss
10,2.858800
20,2.588621
30,2.728300
40,2.824382
50,2.642803
60,2.656781
70,2.435468
80,2.463299
90,2.512478
100,2.423748


TrainOutput(global_step=3762, training_loss=1.4831097093685581, metrics={'train_runtime': 2731.5186, 'train_samples_per_second': 5.509, 'train_steps_per_second': 1.377, 'total_flos': 1.870285948674048e+16, 'train_loss': 1.4831097093685581, 'epoch': 6.0})

In [19]:
print(peft_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PhiForCausalLM(
      (model): PhiModel(
        (embed_tokens): Embedding(51200, 2560)
        (layers): ModuleList(
          (0-31): 32 x PhiDecoderLayer(
            (self_attn): PhiAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=2560, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=2560, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
          

In [20]:
prompt = "Find the describing words for the following sentence:\n\"After they had broken up, he wasn't the same\"."

batch = tokinzer(prompt, return_tensors="pt").to(model.device)

output_tokens = peft_model.generate(
    **batch,
    max_new_tokens=50,
    repetition_penalty=1.1,
)

output = tokinzer.decode(output_tokens[0], skip_special_tokens=True)
print(output)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Find the describing words for the following sentence:
"After they had broken up, he wasn't the same".
Options:
(A) sad
(B) lonely
(C) depressed
(D) heartbroken
(E) miserable
(F) unhappy
(G) sorrowful
(H) tormented
(I)


In [21]:
# Example Before vs After Fine-Tuning (Phi-2)
prompt = "Find the describing words for the following sentence:\n\"After losing his job and going through a tough breakup, he felt completely overwhelmed, lonely, and unsure about the future\"."

# --------- Before Fine-Tuning (Base Phi-2) ---------
batch = tokinzer(prompt, return_tensors="pt").to(model.device)
output_tokens = model.generate(
    **batch,
    max_new_tokens=50,
    repetition_penalty=1.1,
)
output_base = tokinzer.decode(output_tokens[0], skip_special_tokens=True)

# --------- After Fine-Tuning (PEFT Phi-2) ---------
batch = tokinzer(prompt, return_tensors="pt").to(peft_model.device)
output_tokens = peft_model.generate(
    **batch,
    max_new_tokens=50,
    repetition_penalty=1.1,
)
output_peft = tokinzer.decode(output_tokens[0], skip_special_tokens=True)

# --------- Print Comparison ---------
print("="*70)
print("🔹 Prompt:")
print(prompt)
print("="*70)
print("🟥 Before Fine-Tuning (Base Phi-2):")
print(output_base)
print("="*70)
print("🟩 After Fine-Tuning (PEFT Phi-2):")
print(output_peft)
print("="*70)
print("✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the sentence.")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


🔹 Prompt:
Find the describing words for the following sentence:
"After losing his job and going through a tough breakup, he felt completely overwhelmed, lonely, and unsure about the future".
🟥 Before Fine-Tuning (Base Phi-2):
Find the describing words for the following sentence:
"After losing his job and going through a tough breakup, he felt completely overwhelmed, lonely, and unsure about the future".
Tags: feeling, hopelessness, loneliness, life, loss, sadness, uncertainty
"I'm not sure if I can do this anymore. It's just too much."
Tags: ability, confidence, courage, determination, effort, failure
🟩 After Fine-Tuning (PEFT Phi-2):
Find the describing words for the following sentence:
"After losing his job and going through a tough breakup, he felt completely overwhelmed, lonely, and unsure about the future".
Tags: "overwhelmed", "lonely", "future", "job", "breakup"
✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the sentence.

In [22]:
# Rich-based Infographic comparison Before vs After
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()

# Example sentence (longer for clarity)
sentence = "After losing his job and going through a tough breakup, he felt completely overwhelmed, lonely, and unsure about the future."

# Prompt
prompt = f"Find the describing words for the following sentence:\n\"{sentence}\"."

# Generate outputs
batch = tokinzer(prompt, return_tensors="pt").to(model.device)
output_tokens_base = model.generate(**batch, max_new_tokens=50, repetition_penalty=1.1)
output_base = tokinzer.decode(output_tokens_base[0], skip_special_tokens=True)

batch = tokinzer(prompt, return_tensors="pt").to(peft_model.device)
output_tokens_peft = peft_model.generate(**batch, max_new_tokens=50, repetition_penalty=1.1)
output_peft = tokinzer.decode(output_tokens_peft[0], skip_special_tokens=True)

# Display infographic
console.print(Panel(Markdown(f"### 🔹 Prompt\n{prompt}"), border_style="cyan"))

console.print(Panel(Markdown(f"### 🟥 Before Fine-Tuning (Base Phi-2)\n{output_base}"), border_style="red"))

console.print(Panel(Markdown(f"### 🟩 After Fine-Tuning (PEFT Phi-2)\n{output_peft}"), border_style="green"))

console.print(Panel(Markdown("✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the sentence."), border_style="yellow"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                    🔹 Prompt                                                    │
│                                                                                                                 │
│ Find the describing words for the following sentence: "After losing his job and going through a tough breakup,  │
│ he felt completely overwhelmed, lonely, and unsure about the future.".                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                       🟥 Before Fine-Tuning (Base Phi-2)                                        │
│                                                                                                                 │
│ Find the describing words for the following sentence: "After losing his job and going through a tough breakup,  │
│ he felt completely overwhelmed, lonely, and unsure about the future.". Answer: Overwhelmed, lonely, unsure.     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                        🟩 After Fine-Tuning (PEFT Phi-2)                                        │
│                                                                                                                 │
│ Find the describing words for the following sentence: "After losing his job and going through a tough breakup,  │
│ he felt completely overwhelmed, lonely, and unsure about the future.". Options: (A) Job loss, loneliness,       │
│ uncertainty (B) Breakup, overwhelm, loneliness (C) Uncertainty, loneliness, job loss (D) Loneliness, job loss,  │
│ uncertainty (E)                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the   │
│ sentence.                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [23]:
# Rich-based Infographic comparison Before vs After (Second Example)
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()

# Example sentence 2
sentence2 = "She felt anxious and overwhelmed as the deadline approached, knowing that she hadn't finished the project yet."

# Prompt
prompt2 = f"Find the describing words for the following sentence:\n\"{sentence2}\"."

# Generate outputs
batch2 = tokinzer(prompt2, return_tensors="pt").to(model.device)
output_tokens_base2 = model.generate(**batch2, max_new_tokens=50, repetition_penalty=1.1)
output_base2 = tokinzer.decode(output_tokens_base2[0], skip_special_tokens=True)

batch2 = tokinzer(prompt2, return_tensors="pt").to(peft_model.device)
output_tokens_peft2 = peft_model.generate(**batch2, max_new_tokens=50, repetition_penalty=1.1)
output_peft2 = tokinzer.decode(output_tokens_peft2[0], skip_special_tokens=True)

# Display infographic for second example
console.print(Panel(Markdown(f"### 🔹 Prompt\n{prompt2}"), border_style="cyan"))

console.print(Panel(Markdown(f"### 🟥 Before Fine-Tuning (Base Phi-2)\n{output_base2}"), border_style="red"))

console.print(Panel(Markdown(f"### 🟩 After Fine-Tuning (PEFT Phi-2)\n{output_peft2}"), border_style="green"))

console.print(Panel(Markdown("✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the sentence."), border_style="yellow"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                    🔹 Prompt                                                    │
│                                                                                                                 │
│ Find the describing words for the following sentence: "She felt anxious and overwhelmed as the deadline         │
│ approached, knowing that she hadn't finished the project yet.".                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                       🟥 Before Fine-Tuning (Base Phi-2)                                        │
│                                                                                                                 │
│ Find the describing words for the following sentence: "She felt anxious and overwhelmed as the deadline         │
│ approached, knowing that she hadn't finished the project yet.". Answer: The describing words are: "anxious",    │
│ "overwhelmed", "deadline", "project".                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                        🟩 After Fine-Tuning (PEFT Phi-2)                                        │
│                                                                                                                 │
│ Find the describing words for the following sentence: "She felt anxious and overwhelmed as the deadline         │
│ approached, knowing that she hadn't finished the project yet.". Answer: ['anxious', 'overwhelmed']              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the   │
│ sentence.                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
# Rich-based Infographic comparison Before vs After (Second Example)
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()

# Example sentence 2
sentence2 = "She felt anxious and overwhelmed as the deadline approached, knowing that she hadn't finished the project yet."

# Prompt
prompt2 = f"Find the describing words for the following sentence:\n\"{sentence2}\"."

# Generate outputs
batch2 = tokinzer(prompt2, return_tensors="pt").to(model.device)
output_tokens_base2 = model.generate(**batch2, max_new_tokens=50, repetition_penalty=1.1)
output_base2 = tokinzer.decode(output_tokens_base2[0], skip_special_tokens=True).replace(prompt2, "")

batch2 = tokinzer(prompt2, return_tensors="pt").to(peft_model.device)
output_tokens_peft2 = peft_model.generate(**batch2, max_new_tokens=50, repetition_penalty=1.1)
output_peft2 = tokinzer.decode(output_tokens_peft2[0], skip_special_tokens=True).replace(prompt2, "")

# Display infographic for second example
console.print(Panel(Markdown(f"### 🔹 Prompt\n{prompt2}"), border_style="cyan"))

console.print(Panel(Markdown(f"### 🟥 Before Fine-Tuning (Base Phi-2)\n{output_base2}"), border_style="red"))

console.print(Panel(Markdown(f"### 🟩 After Fine-Tuning (PEFT Phi-2)\n{output_peft2}"), border_style="green"))

console.print(Panel(Markdown("✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the sentence."), border_style="yellow"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                    🔹 Prompt                                                    │
│                                                                                                                 │
│ Find the describing words for the following sentence: "She felt anxious and overwhelmed as the deadline         │
│ approached, knowing that she hadn't finished the project yet.".                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                       🟥 Before Fine-Tuning (Base Phi-2)                                        │
│                                                                                                                 │
│ A: The describing words are: anxious, overwhelmed, deadline, project.                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                        🟩 After Fine-Tuning (PEFT Phi-2)                                        │
│                                                                                                                 │
│ Answer: Anxious, overwhelmed, deadline, project.                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the   │
│ sentence.                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [25]:
# Rich-based Infographic comparison Before vs After (Third Example)
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()

# Example sentence 3 (more complex & longer)
sentence3 = "After months of struggling to meet deadlines, dealing with constant criticism from his manager, and feeling isolated from his colleagues, he became frustrated, anxious, and demotivated."

# Prompt
prompt3 = f"Find the describing words for the following sentence:\n\"{sentence3}\"."

# Generate outputs
batch3 = tokinzer(prompt3, return_tensors="pt").to(model.device)
output_tokens_base3 = model.generate(**batch3, max_new_tokens=60, repetition_penalty=1.1)
output_base3 = tokinzer.decode(output_tokens_base3[0], skip_special_tokens=True).replace(prompt3, "")

batch3 = tokinzer(prompt3, return_tensors="pt").to(peft_model.device)
output_tokens_peft3 = peft_model.generate(**batch3, max_new_tokens=60, repetition_penalty=1.1)
output_peft3 = tokinzer.decode(output_tokens_peft3[0], skip_special_tokens=True).replace(prompt3, "")

# Display infographic for third example
console.print(Panel(Markdown(f"### 🔹 Prompt\n{prompt3}"), border_style="cyan"))

console.print(Panel(Markdown(f"### 🟥 Before Fine-Tuning (Base Phi-2)\n{output_base3}"), border_style="red"))

console.print(Panel(Markdown(f"### 🟩 After Fine-Tuning (PEFT Phi-2)\n{output_peft3}"), border_style="green"))

console.print(Panel(Markdown("✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the sentence."), border_style="yellow"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                    🔹 Prompt                                                    │
│                                                                                                                 │
│ Find the describing words for the following sentence: "After months of struggling to meet deadlines, dealing    │
│ with constant criticism from his manager, and feeling isolated from his colleagues, he became frustrated,       │
│ anxious, and demotivated.".                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                       🟥 Before Fine-Tuning (Base Phi-2)                                        │
│                                                                                                                 │
│ Tags: anxiety, boss, frustration, isolation, motivation, stress                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                        🟩 After Fine-Tuning (PEFT Phi-2)                                        │
│                                                                                                                 │
│ Tags: anxiety, frustration, isolation, motivation, stress                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ Observation: After fine-tuning, the model extracts descriptive keywords correctly instead of repeating the   │
│ sentence.                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# **Save Model**

In [26]:
peft_model.save_pretrained("peft_model_phi-2")

# **Model Loading**

In [27]:
device = "cuda" if torch.cuda.is_available() else "cpu"
config = PeftConfig.from_pretrained("peft_model_phi-2")
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
model.to(device)
lora_model = PeftModel.from_pretrained(model, "peft_model_phi-2")

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [28]:
# Rich-based Infographic comparison Before vs After
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()

# Example sentence (longer for clarity)
sentence = "He is a good kickboxing and MMA player, but he struggles with motivation and confidence in competitions."

# Improved Prompt with Example
prompt = f"""
Extract only descriptive keywords that summarize the person's skills, personality, or mood.
Example: "She is confident and hardworking." -> ['confident', 'hardworking']
Sentence: "{sentence}"
"""

# Generate outputs from Base Model
batch = tokinzer(prompt, return_tensors="pt").to(model.device)
output_tokens_base = model.generate(**batch, max_new_tokens=50, repetition_penalty=1.1)
output_base = tokinzer.decode(output_tokens_base[0], skip_special_tokens=True).replace(prompt, "")

# Generate outputs from PEFT / LoRA Model
batch = tokinzer(prompt, return_tensors="pt").to(lora_model.device)
output_tokens_peft = lora_model.generate(**batch, max_new_tokens=50, repetition_penalty=1.1)
output_peft = tokinzer.decode(output_tokens_peft[0], skip_special_tokens=True).replace(prompt, "")

# Display infographic
console.print(Panel(Markdown(f"### 🔹 Prompt\n{prompt.strip()}"), border_style="cyan"))
console.print(Panel(Markdown(f"### 🟥 Before Fine-Tuning (Base Phi-2)\n{output_base.strip()}"), border_style="red"))
console.print(Panel(Markdown(f"### 🟩 After Fine-Tuning (PEFT Phi-2)\n{output_peft.strip()}"), border_style="green"))
console.print(Panel(Markdown(
    "✅ Observation: After fine-tuning, the PEFT model extracts descriptive keywords correctly instead of repeating the sentence."
), border_style="yellow"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                    🔹 Prompt                                                    │
│                                                                                                                 │
│ Extract only descriptive keywords that summarize the person's skills, personality, or mood. Example: "She is    │
│ confident and hardworking." -> ['confident', 'hardworking'] Sentence: "He is a good kickboxing and MMA player,  │
│ but he struggles with motivation and confidence in competitions."                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                       🟥 Before Fine-Tuning (Base Phi-2)                                        │
│                                                                                                                 │
│ Tags: ['kickboxing','mixed-martial-arts','motivation','self-confidence']                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                        🟩 After Fine-Tuning (PEFT Phi-2)                                        │
│                                                                                                                 │
│ Tags: ['kickboxing','mixed-martial-arts','motivation','self-confidence']                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ Observation: After fine-tuning, the PEFT model extracts descriptive keywords correctly instead of repeating  │
│ the sentence.                                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯